# 3.4 Structural Pattern Matching (match / case)

**Prerequisites:** 3.1 Decision Statement, 2.3 Tuple, 2.5 Dictionary  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What `match`/`case` is — and why it is not a switch statement
- Literal, capture and wildcard patterns
- Sequence patterns, including `*rest`
- Mapping patterns for dispatching on dict shape
- Class patterns, or-patterns (`|`), `as` patterns and `if` guards
- The capture-vs-comparison trap
- When `match` genuinely beats `if`/`elif` — and when it doesn't

---

## Why this notebook is new

> **Version note:** `match` / `case` was added in **Python 3.10** (PEP 634), in 2021.
> These notes were written in 2019, so it appears nowhere in the original material. It is
> the single largest piece of new *syntax* Python has gained since then.

It is also the most commonly misunderstood. Almost every tutorial introduces it as
"Python finally got a switch statement". That framing is wrong and will lead you to write
bad `match` statements.

### It is not a switch

A C-style `switch` compares one value against a list of constants. `match` compares a value
against **structural patterns** — shapes. It can ask:

- Is this a list of exactly three things, and what are they?
- Is this a dict with a `"type"` key equal to `"circle"`, and what is its `"radius"`?
- Is this a `Point` object, and what are its `x` and `y`?

...and **bind the pieces to names** in the same step. That destructuring is the point. If
all you need is equality against a few constants, a dict lookup or an `if`/`elif` chain is
simpler, and you should use one.

---

## 1. Syntax breakdown

```
match subject:
    case pattern_1:
        ...
    case pattern_2 if guard:
        ...
    case _:
        ...
```

| Part | Meaning |
|---|---|
| `subject` | The value being examined — any expression |
| `case pattern:` | Tried **in order**, top to bottom |
| `if guard` | An extra condition; the case matches only if the pattern *and* the guard hold |
| `case _:` | The **wildcard** — matches anything. Conventionally last, as a catch-all |

Only the **first** matching case runs. There is no fall-through, so no `break` is needed.

`match` and `case` are **soft keywords** (see **1.2**) — you can still use `match` as a
variable name.

In [ ]:
def http_status(code):
    match code:
        case 200:
            return "OK"
        case 301 | 302 | 307:          # or-pattern: any of these
            return "Redirect"
        case 400:
            return "Bad Request"
        case 404:
            return "Not Found"
        case _:                         # wildcard: anything else
            return f"Unhandled status {code}"


for code in [200, 302, 404, 500]:
    print(f"  {code}: {http_status(code)}")

# No fall-through: only the first match runs, and no `break` is needed.
# `match` is a STATEMENT, not an expression - it does not produce a value,
# which is why these examples use `return`.

---

## 2. ⚠️ The trap: a bare name always *captures*

This is the mistake everyone makes once.

```python
OK = 200

match status:
    case OK:          # does NOT compare against 200!
        ...
```

A bare name in a pattern is a **capture pattern** — it binds the subject to that name and
**always matches**. It does not compare. So the case above matches *everything* — and, like
any assignment, rebinds `OK` **in the scope where the `match` statement runs**: at
module level it clobbers the module-level `OK`; inside a function it quietly creates a
function-local `OK` that shadows it (the global survives, but the wrong branch still
ran). The demo below shows both.

### How much does Python protect you?

Partly, and it is worth knowing exactly where the safety net ends:

| Situation | What happens |
|---|---|
| A capture pattern with **other cases after it** | **`SyntaxError` at compile time:** *"name capture 'OK' makes remaining patterns unreachable"*. The file will not even load. |
| A capture pattern that is the **last (or only)** case | **Compiles and runs silently.** It matches everything, and you get the wrong branch with no warning. |

So the dangerous version is the one where the mistake happens to sit at the bottom of your
`match`. That is the case to watch for.

### The fix

To compare against a constant, the pattern must be a **literal** or a **dotted name**:

| Pattern | Behaviour |
|---|---|
| `case 200:` | Literal — compares |
| `case OK:` | **Capture** — always matches, rebinds `OK` |
| `case Status.OK:` | **Value pattern** — compares (dotted name) |
| `case _:` | Wildcard — matches, binds nothing |

This is a large part of why `Enum` and `match` are so often used together: an enum member
is always a dotted name, so it can never be misread as a capture.

In [ ]:
from enum import Enum

OK = 200

# ---- Version 1: Python REFUSES to compile this ----
bad_source = """
match status:
    case OK:          # bare name - captures everything
        result = "matched OK"
    case 404:         # can never be reached
        result = "not found"
"""

try:
    compile(bad_source, "<demo>", "exec")
except SyntaxError as exc:
    print("Python rejects it:", exc.msg)


# ---- Version 2: the SILENT trap - capture as the last case ----
def broken(status):
    match status:
        case 404:
            return "not found"
        case OK:              # compiles fine, matches EVERYTHING
            return f"matched OK (local OK is now {OK})"

print("\nbroken(404) ->", broken(404), "  (correct, by luck)")
print("broken(500) ->", broken(500), "  <- WRONG branch, chosen silently")
print("module OK   ->", OK, "  <- UNCHANGED: inside a function, the capture bound a LOCAL OK")


# ---- Version 2b: the same match at MODULE level - now the global IS rebound ----
status = 500
match status:
    case 404:
        print("not found")
    case OK:              # capture at module scope assigns to the module-level OK
        print(f"\nmodule-level match on {status} -> matched OK")

print("module OK   ->", OK, "  <- the pattern REBOUND the module-level OK to 500")
OK = 200                  # restore it for the rest of the notebook

# ⚠️ Scoping nuance: a capture pattern is an ordinary assignment in whatever scope
# the `match` statement runs. At module level it clobbers the global; inside a
# function it creates/rebinds a function-LOCAL name and the global survives.
# Either way the wrong branch ran - the local case just hides the evidence.


# ---- Fix 1: use a literal ----
def fixed_literal(status):
    match status:
        case 200:
            return "matched OK"
        case 404:
            return "not found"
        case _:
            return "other"


# ---- Fix 2: a dotted name. An Enum is the idiomatic choice. ----
class Status(Enum):
    OK = 200
    NOT_FOUND = 404


def fixed_enum(status):
    match Status(status):
        case Status.OK:              # dotted -> compares, never captures
            return "matched OK"
        case Status.NOT_FOUND:
            return "not found"


print("\nfixed_literal(500) ->", fixed_literal(500))
print("fixed_enum(200)    ->", fixed_enum(200))
print("fixed_enum(404)    ->", fixed_enum(404))

---

## 3. Sequence patterns

This is where `match` starts doing things `if`/`elif` cannot do in one step: check the
**shape** of a sequence and pull out its parts simultaneously.

```
case [x, y]:        a sequence of exactly 2 -> bind them to x and y
case [x, *rest]:    at least 1 -> bind the first, collect the rest as a list
case []:            an empty sequence
case (x, y):        parentheses and brackets are interchangeable here
```

A sequence pattern matches lists and tuples — but **not** strings, `bytes`, or a plain
iterator. That exclusion is deliberate: matching `"ab"` as a two-element sequence would
almost never be what you wanted.

In [ ]:
def describe(seq):
    match seq:
        case []:
            return "empty"
        case [x]:
            return f"one item: {x}"
        case [x, y]:
            return f"two items: {x} and {y}"
        case [x, y, z]:
            return f"three items: {x}, {y}, {z}"
        case [first, *rest]:
            return f"{len(rest) + 1} items, starting with {first}, rest={rest}"
        case _:
            return "not a sequence"


for value in [[], [1], [1, 2], [1, 2, 3], [1, 2, 3, 4, 5], (9, 8), "ab", 42]:
    print(f"  {str(value):<16} -> {describe(value)}")

print("\nNote: 'ab' is NOT matched as a 2-sequence - strings are excluded on purpose.")


# Nested sequence patterns destructure several levels at once
def parse_point_pair(data):
    match data:
        case [[x1, y1], [x2, y2]]:
            return f"segment from ({x1},{y1}) to ({x2},{y2})"
        case [[x, y]]:
            return f"single point ({x},{y})"
        case _:
            return "unrecognised"

print("\n" + parse_point_pair([[0, 0], [3, 4]]))
print(parse_point_pair([[1, 2]]))

---

## 4. Mapping patterns

Mapping patterns match dictionaries by the keys they contain.

```
case {"type": "circle", "radius": r}:
```

Two important behaviours:

1. **Mapping patterns match on a subset.** A dict with *extra* keys still matches — unlike
   sequence patterns, which require an exact length. This is what makes them practical for
   JSON, where payloads carry fields you don't care about.
2. **Use `**rest`** to capture the leftover keys if you need them.

**Real-world use case:** handling a JSON API response or an event payload, where the
meaningful branch depends on the value of a `"type"` or `"event"` field.

In [ ]:
import math

def area(shape):
    match shape:
        case {"type": "circle", "radius": r}:
            return math.pi * r ** 2
        case {"type": "rectangle", "width": w, "height": h}:
            return w * h
        case {"type": "square", "side": s}:
            return s ** 2
        case {"type": name}:
            raise ValueError(f"unknown shape: {name}")
        case _:
            raise TypeError("not a shape dict")


shapes = [
    {"type": "circle", "radius": 2},
    {"type": "rectangle", "width": 3, "height": 4},
    {"type": "square", "side": 5, "colour": "red"},   # extra key - still matches
]

for s in shapes:
    print(f"  {s['type']:<10} area = {area(s):.2f}")

try:
    area({"type": "hexagon"})
except ValueError as exc:
    print("\n", exc)


# **rest captures the keys you did not name
def inspect(payload):
    match payload:
        case {"event": kind, **extra}:
            return f"event={kind}, other fields={extra}"
        case _:
            return "no event field"

print("\n" + inspect({"event": "login", "user": "aditya", "ip": "10.0.0.1"}))

---

## 5. Class patterns

Class patterns check the type **and** destructure attributes in one step.

```
case Point(x=0, y=0):        a Point whose x and y are both 0
case Point(x=px, y=py):      any Point; bind its x and y
case Point(0, 0):            positional - only if the class defines __match_args__
```

Positional class patterns need `__match_args__`, which tells Python which attributes the
positional slots refer to. **`dataclass` and `NamedTuple` generate it for you** — which is
a large part of why they pair so well with `match`.

(Classes are covered in **05 OOPs**; dataclasses in **5.3 Dataclasses and Enums**.)

In [ ]:
from dataclasses import dataclass


@dataclass
class Point:
    x: float
    y: float


@dataclass
class Circle:
    centre: Point
    radius: float


def locate(obj):
    match obj:
        case Point(x=0, y=0):
            return "origin"
        case Point(x=0, y=y):
            return f"on the y-axis at {y}"
        case Point(x=x, y=0):
            return f"on the x-axis at {x}"
        case Point(x=x, y=y):
            return f"at ({x}, {y})"
        case Circle(centre=Point(x=0, y=0), radius=r):
            return f"circle centred on the origin, radius {r}"
        case Circle(centre=c, radius=r):
            return f"circle at ({c.x}, {c.y}), radius {r}"
        case _:
            return "unknown object"


for obj in [
    Point(0, 0),
    Point(0, 5),
    Point(3, 0),
    Point(3, 4),
    Circle(Point(0, 0), 10),
    Circle(Point(1, 1), 2),
    "not a shape",
]:
    print(f"  {str(obj):<34} -> {locate(obj)}")

# @dataclass generates __match_args__, so positional patterns work too
print("\n__match_args__ :", Point.__match_args__)

match Point(7, 8):
    case Point(a, b):                 # positional
        print(f"positional pattern bound a={a}, b={b}")

---

## 6. Guards, `as` patterns, and or-patterns

Three modifiers that make patterns precise:

| Feature | Syntax | Purpose |
|---|---|---|
| **Guard** | `case [x, y] if x > y:` | Extra condition, evaluated *after* the pattern matches |
| **`as` pattern** | `case [Point() as p, _]:` | Match a sub-pattern **and** keep the whole thing |
| **Or-pattern** | `case 301 \| 302 \| 307:` | Any one of several alternatives |

A guard is **not** part of the pattern — it runs only if the structure matched, so any
names bound by the pattern are available to it.

In [ ]:
def classify(pair):
    match pair:
        case [x, y] if x == y:
            return f"equal ({x})"
        case [x, y] if x > y:
            return f"descending ({x} > {y})"
        case [x, y]:
            return f"ascending ({x} < {y})"
        case _:
            return "not a pair"


for p in [[1, 1], [5, 2], [2, 5], [1, 2, 3]]:
    print(f"  {str(p):<12} -> {classify(p)}")


# `as` keeps the matched object while still checking its shape
def first_valid(items):
    match items:
        case [{"ok": True} as good, *_]:
            return f"first item is valid: {good}"
        case [_, *rest] if rest:
            return "first item invalid, but there are more"
        case _:
            return "nothing usable"

print("\n" + first_valid([{"ok": True, "id": 1}, {"ok": False}]))
print(first_valid([{"ok": False}, {"ok": True}]))


# Or-patterns can bind, as long as EVERY alternative binds the same names
def direction(cmd):
    match cmd:
        case "n" | "north":
            return "going north"
        case ["go", ("n" | "north") as where]:
            return f"going {where}"
        case _:
            return "?"

print("\n" + direction("north"))
print(direction(["go", "n"]))

---

## 7. A real example: a command parser

This is the shape of problem `match` was designed for — input whose *structure* varies, not
just its value. Written with `if`/`elif` this needs length checks, index guards and manual
unpacking on every branch.

In [ ]:
def run(command):
    """Interpret a command given as a list of words."""
    match command.split():
        case []:                       # most specific first - "" splits to []
            return "Say something"
        case ["quit" | "exit"]:
            return "Goodbye"
        case ["help"]:
            return "Commands: quit, help, go <dir>, get <item> [count], set <k> = <v>"
        case ["go", direction]:
            return f"Walking {direction}"
        case ["get", item]:
            return f"Picked up 1 x {item}"
        case ["get", item, count] if count.isdigit():
            return f"Picked up {count} x {item}"
        case ["set", key, "=", value]:
            return f"{key} is now {value}"
        case [verb, *args]:    # needs >= 1 word, so it could never steal []
            return f"Unknown command {verb!r} with args {args}"


for cmd in [
    "quit", "help", "go north", "get lamp", "get coin 5",
    "set volume = 11", "dance wildly now", "",
]:
    print(f"  {cmd!r:<22} -> {run(cmd)}")

---

## 8. When to use `match` — and when not to

`match` is a specialised tool. Reach for it when the **structure** of the data drives the
logic; avoid it when a simpler construct says the same thing.

| Situation | Best tool |
|---|---|
| Map a value to another value | **dict lookup** |
| Test membership in a set of options | **`in`** |
| Two or three unrelated boolean conditions | **`if` / `elif`** |
| Dispatch on the *shape* of a sequence, dict or object | **`match`** |
| Destructure while branching | **`match`** |
| Parse commands, tokens, ASTs, JSON payloads | **`match`** |
| Implement a state machine over structured events | **`match`** |

Two practical constraints worth remembering:

- `match` is a **statement**, not an expression. It doesn't produce a value, so you'll
  usually `return` from inside it or assign in each branch.
- It requires **Python 3.10+**. If your code must run on 3.9, you cannot use it at all.

In [ ]:
# The same problem, both ways - so you can judge for yourself.

events = [
    {"type": "click", "pos": [10, 20]},
    {"type": "keypress", "key": "a", "modifiers": ["ctrl"]},
    {"type": "scroll", "delta": -3},
    {"type": "unknown"},
]

# ---- if/elif: every branch re-checks and re-indexes ----
def handle_if(e):
    if e.get("type") == "click" and isinstance(e.get("pos"), list) and len(e["pos"]) == 2:
        x, y = e["pos"]
        return f"click at ({x}, {y})"
    elif e.get("type") == "keypress" and "ctrl" in e.get("modifiers", []):
        return f"ctrl+{e['key']}"
    elif e.get("type") == "keypress":
        return f"key {e['key']}"
    elif e.get("type") == "scroll":
        return f"scroll {e['delta']}"
    else:
        return "unhandled"


# ---- match: the shape IS the condition ----
def handle_match(e):
    match e:
        case {"type": "click", "pos": [x, y]}:
            return f"click at ({x}, {y})"
        case {"type": "keypress", "key": k, "modifiers": ["ctrl", *_]}:
            return f"ctrl+{k}"
        case {"type": "keypress", "key": k}:
            return f"key {k}"
        case {"type": "scroll", "delta": d}:
            return f"scroll {d}"
        case _:
            return "unhandled"


for e in events:
    a, b = handle_if(e), handle_match(e)
    assert a == b, (a, b)
    print(f"  {str(e)[:44]:<46} -> {b}")

print("\nIdentical results. The match version states the shape it expects;")
print("the if version reconstructs that shape from primitives on every branch.")

---

## Common Mistakes & Pitfalls

1. **Using a bare name as a constant pattern.** `case OK:` captures — it matches everything and rebinds `OK` in whatever scope the `match` runs (a *local* `OK`, if inside a function). Python raises `SyntaxError` only if later cases become unreachable; as the *last* case it fails silently. Use a literal or a dotted name (`Status.OK`).
2. **Treating it as a switch.** If every case is `case <literal>:` and there is no destructuring, a dict lookup is simpler and faster to read.
3. **Expecting fall-through.** Only the first matching case runs; there is no `break`, and adding one is a `SyntaxError` outside a loop.
4. **Expecting a string to match a sequence pattern.** `str` and `bytes` are deliberately excluded, so `case [a, b]:` will not match `"ab"`.
5. **Forgetting that sequence patterns need an exact length** (unless you use `*rest`), while **mapping patterns match on a subset** and ignore extra keys.
6. **Putting `case _:` before other cases.** It matches everything, so everything after it is dead code.
7. **Using positional class patterns without `__match_args__`.** Plain classes raise `TypeError`; use keyword patterns, or a `dataclass`/`NamedTuple`.
8. **Assuming it works on Python 3.9.** It is 3.10+, and the failure is a `SyntaxError` at import time — the whole file fails to load.

## Best Practices

- Use `match` for **structure**, `if`/`elif` for conditions, dicts for value mapping.
- Put the most specific patterns first and the wildcard `case _:` last.
- Use `Enum` members or literals for constant comparison — never bare names.
- Pair `match` with `dataclass` / `NamedTuple`: they generate `__match_args__` and make class patterns read cleanly.
- Use guards for conditions that aren't structural (`if x > y`), not for shape checks.
- Always include a `case _:` unless you genuinely want unmatched input to fall through silently doing nothing.
- Keep patterns shallow — three levels of nesting is usually a sign the data wants a class.

## Practice Exercises

Try these before moving on.

1. Write a `match` that classifies a value as empty list, single item, pair, or longer.
2. Handle a JSON-like dict with `{'status': 'ok', 'data': [...]}` versus `{'status': 'error', 'message': ...}` and return a readable summary of each.
3. Extend the command parser above with `drop <item>` and `look`.
4. Write a calculator that matches `[a, '+', b]`, `[a, '-', b]`, etc., with a guard preventing division by zero.
5. Define a `Shape` hierarchy with dataclasses and compute area using class patterns.
6. Demonstrate the capture-pattern trap: write a `match` that silently does the wrong thing, then fix it two ways.
7. Take an `if`/`elif` chain from **3.1** and decide whether `match` would improve it. Justify your answer either way.